## परिचय 

हा धडा खालील विषयांवर आहे: 
- फंक्शन कॉलिंग म्हणजे काय आणि त्याचे वापरप्रकरणे 
- OpenAI वापरून फंक्शन कॉल कसे तयार करायचे 
- फंक्शन कॉल कसा अॅप्लिकेशनमध्ये समाकलित करायचा 

## शिकण्याचे उद्दिष्टे 

हा धडा पूर्ण केल्यानंतर तुम्हाला कसे करायचे हे कळेल आणि समजून येईल: 

- फंक्शन कॉलिंगचा उद्देश 
- OpenAI सेवेचा वापर करून फंक्शन कॉल सेटअप करणे 
- तुमच्या अॅप्लिकेशनच्या वापरप्रकरणासाठी प्रभावी फंक्शन कॉल डिझाइन करणे 


## फंक्शन कॉल समजून घेणे

या धड्यातील, आम्हाला आमच्या शिक्षण स्टार्टअपसाठी एक अशी सुविधा तयार करायची आहे ज्यामुळे वापरकर्ते तांत्रिक कोर्स शोधण्यासाठी चॅटबॉटचा वापर करू शकतील. आम्ही अशा कोर्सची शिफारस करू जे त्यांच्या कौशल्य पातळी, सध्याच्या भूमिकेप्रमाणे आणि आवडत्या तंत्रज्ञानानुसार जुळतील.

हे पूर्ण करण्यासाठी आम्ही खालील संयोजन वापरणार आहोत:
 - वापरकर्त्यासाठी चॅट अनुभव तयार करण्यासाठी `OpenAI`
 - वापरकर्त्यांच्या विनंतीनुसार कोर्स शोधायला मदत करण्यासाठी `Microsoft Learn Catalog API`
 - वापरकर्त्याच्या चौकशीला घेऊन API विनंती करण्यासाठी `Function Calling`

सुरवात करण्यासाठी, पाहूया की आपल्याला फंक्शन कॉलिंग का वापरायचे आहे:


### फंक्शन कॉलिंग का आवश्यक आहे

जर आपण या कोर्समधील इतर कोणताही धडा पूर्ण केला असेल, तर तुम्हाला मोठ्या भाषा मॉडेल्स (LLMs) वापरण्याची क्षमता समजली असेल. आशा आहे की तुम्ही त्यांच्या काही मर्यादा देखील पाहिल्या असतील.

फंक्शन कॉलिंग ही OpenAI सेवा वैशिष्ट्य आहे जी खालील अडचणींवर मात करण्यासाठी डिझाईन केलेली आहे:

असमान प्रतिसाद स्वरूप:
- फंक्शन कॉलिंगपूर्वी, मोठ्या भाषा मॉडेलकडून मिळणारे प्रतिसाद अनियमित आणि असमान होते. विकासकांनी प्रत्येक विविधतेसाठी जटिल वैधता कोड लिहावा लागायचा.

बाह्य डेटासोबत मर्यादित एकत्रीकरण:
- या वैशिष्ट्यापूर्वी, चॅट संदर्भात ऍप्लिकेशनच्या इतर भागांमधील डेटा समाविष्ट करणे कठीण होते.

प्रतिसाद स्वरूपांची मानकीकरण आणि बाह्य डेटासोबत सुलभ एकत्रीकरण सक्षम करून, फंक्शन कॉलिंग विकास सुलभ करते आणि अतिरिक्त वैधता लॉजिकची आवश्यकता कमी करते.

वापरकर्त्यांना "स्टॉकहोममधले सध्याचे हवामान काय आहे?" असे प्रश्नांची उत्तरे मिळू शकत नव्हती. कारण मॉडेल त्यासाठी फक्त प्रशिक्षण घेतलेल्या डेटापर्यंत मर्यादित होते.

खाली दिलेला उदाहरण पाहूया जो ही समस्या स्पष्ट करतो:

समजा आपण विद्यार्थी डेटाचा एक डेटाबेस तयार करू इच्छितो जेणेकरून आपण त्यांना योग्य कोर्स सुचवू शकू. खाली दोन विद्यार्थ्यांचे वर्णन आहे ज्यामध्ये डेटा जवळजवळ सारखाच आहे.


In [ ]:
student_1_description="Emily Johnson is a sophomore majoring in computer science at Duke University. She has a 3.7 GPA. Emily is an active member of the university's Chess Club and Debate Team. She hopes to pursue a career in software engineering after graduating."
 
student_2_description = "Michael Lee is a sophomore majoring in computer science at Stanford University. He has a 3.8 GPA. Michael is known for his programming skills and is an active member of the university's Robotics Club. He hopes to pursue a career in artificial intelligence after finishing his studies."


आम्हाला हे एक LLM कडे डेटा पार्स करण्यासाठी पाठवायचे आहे. नंतर हे आमच्या अनुप्रयोगात API कडे पाठवण्यासाठी किंवा डेटाबेसमध्ये संग्रहित करण्यासाठी वापरले जाऊ शकते.

चला दोन सारखे प्रॉम्प्ट तयार करू जे आम्ही LLM ला निर्देशित करू की आम्हाला कोणती माहिती लागेल:


आम्हाला हे LLM कडे पाठवायचे आहे जेणेकरून आमच्या उत्पादनासाठी महत्त्वाच्या भागांचे विश्लेषण करता येईल. म्हणून आम्ही LLM ला निर्देश देण्यासाठी दोन समान सुचनाअर्थ तयार करू शकतो:


In [ ]:
prompt1 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_1_description}
'''


prompt2 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_2_description}
'''


हे दोन प्रॉम्प्ट तयार केल्यानंतर, आम्ही त्यांना `client.responses.create` वापरून LLM कडे पाठवू. आम्ही प्रॉम्प्ट `input` व्हेरियबलमध्ये साठवतो आणि रोल `user` ला असाइन करतो. हे वापरकर्त्याच्या संदेशाला चॅटबॉटमध्ये लिहिण्याचे अनुकरण करण्यासाठी आहे.



In [ ]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

client = OpenAI()

deployment="gpt-4o-mini"


: 

आता आपण दोन्ही विनंत्या LLM कडे पाठवू शकतो आणि आपल्याला मिळणारी प्रतिक्रिया तपासू शकतो. 


In [ ]:
openai_response1 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt1}],
 store=False,
)
openai_response1.output_text 


In [ ]:
openai_response2 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt2}],
 store=False,
)
openai_response2.output_text


In [ ]:
# Loading the response as a JSON object
json_response1 = json.loads(openai_response1.output_text)
json_response1


In [ ]:
# Loading the response as a JSON object
json_response2 = json.loads(openai_response2.output_text)
json_response2



जरी प्रम्प्ट एकसमान असले तरी आणि वर्णन सादृश्य असले तरी, आपण `Grades` गुणधर्माचे वेगवेगळे स्वरूप मिळवू शकतो.

जर आपण वरील सेल अनेक वेळा चालवले, तर स्वरूप `3.7` किंवा `3.7 GPA` असू शकते.

याचे कारण म्हणजे LLM लिहिलेल्या प्रम्प्टच्या रूपात असंरचित डेटा घेते आणि परतही असंरचित डेटा देते. आपल्याला संरचित स्वरूप आवश्यक आहे जेणेकरून आपल्याला समजेल की हे डेटा संग्रहित किंवा वापरताना काय अपेक्षा करावी.

फंक्शनल कॉलिंग वापरून, आपण सुनिश्चित करू शकतो की आपल्याला संरचित डेटा मिळेल. फंक्शन कॉलिंगचा वापर करताना, LLM प्रत्यक्षात कोणतेही फंक्शन कॉल किंवा चालवत नाही. त्याऐवजी, आपण LLM साठी त्याच्या प्रतिसादांसाठी एक संरचना तयार करतो. नंतर आपण त्या संरचित प्रतिसादांचा वापर करून आमच्या अनुप्रयोगांमध्ये कोणते फंक्शन चालवायचे ते ठरवतो.
 


![फंक्शन कॉलिंग फ्लो डायग्राम](../../../../translated_images/mr/Function-Flow.083875364af4f4bb.webp)


आपण नंतर फंक्शनमधून परत येणाऱ्या गोष्टी घेतल्या आणि हे LLM कडे परत पाठवू शकतो. नंतर LLM नैसर्गिक भाषेचा वापर करून वापरकर्त्याच्या प्रश्नाचे उत्तर देईल.


### फंक्शन कॉल वापरण्याचे वापर प्रकरणे 

**बाह्य टूल्स कॉल करणे** 
चॅटबॉट्स वापरकर्त्यांच्या प्रश्नांची उत्तरे देण्यात उत्कृष्ट असतात. फंक्शन कॉलिंग वापरून, चॅटबॉट्स वापरकर्त्यांकडून आलेल्या संदेशांचा वापर विशिष्ट कार्ये पूर्ण करण्यासाठी करू शकतात. उदाहरणार्थ, एक विद्यार्थी चॅटबॉटला विचारू शकतो "माझ्या शिक्षकाला ईमेल पाठवा की मला या विषयात अधिक मदतीची गरज आहे". ही एक फंक्शन कॉल करू शकते `send_email(to: string, body: string)`


**API किंवा डेटाबेस क्वेरी तयार करणे**
वापरकर्ते नैसर्गिक भाषेचा वापर करून माहिती शोधू शकतात जी एका स्वरूपित क्वेरी किंवा API विनंतीत रूपांतरित केली जाते. याचे उदाहरण म्हणजे एक शिक्षक जो विचारतो "कोणते विद्यार्थी शेवटचे असाइनमेंट पूर्ण केले आहेत?" आणि एक फंक्शन कॉल करू शकतो ज्याचे नाव आहे `get_completed(student_name: string, assignment: int, current_status: string)`


**रचनाबद्ध डेटा तयार करणे**
वापरकर्ते मजकूराचा किंवा CSV चा ब्लॉक घेऊन LLM चा वापर करून त्यातून महत्त्वाची माहिती काढू शकतात. उदाहरणार्थ, एक विद्यार्थी Wikipedia लेख शांती करारांविषयी रूपांतरित करून AI फ्लॅश कार्ड्स तयार करू शकतो. हे `get_important_facts(agreement_name: string, date_signed: string, parties_involved: list)` नावाच्या फंक्शनचा वापर करून केले जाऊ शकते.


## 2. आपला पहिला फंक्शन कॉल तयार करणे 

फंक्शन कॉल तयार करण्याच्या प्रक्रियेत 3 मुख्य टप्पे असतात: 
1. आपल्या फंक्शन्सची यादी आणि वापरकर्त्याचा संदेश वापरून Chat Completions API कॉल करणे 
2. मॉडेलच्या प्रतिसादाचा अभ्यास करून क्रिया करणे म्हणजे फंक्शन किंवा API कॉल अंमलात आणणे 
3. आपल्या फंक्शनकडून मिळालेल्या प्रतिसादासह Chat Completions API ला पुन्हा कॉल करणे जेणेकरून त्या माहितीस वापरून वापरकर्त्यासाठी प्रतिसाद तयार करता येईल. 


![फंक्शन कॉलचा प्रवाह](../../../../translated_images/mr/LLM-Flow.3285ed8caf4796d7.webp)


### फंक्शन कॉलचे घटक

#### वापरकर्त्याचा इनपुट

पहिला टप्पा म्हणजे वापरकर्ता संदेश तयार करणे. हा संदेश डायनॅमिकली टेक्स्ट इनपुटच्या किमतीने दिला जाऊ शकतो किंवा तुम्ही येथे एक मूल्य देऊ शकता. जर तुम्ही Chat Completions API सोबत प्रथम वेळ काम करत असाल, तर आपल्याला संदेशाचे `role` आणि `content` परिभाषित करणे आवश्यक आहे.

`role` हे `system` (नियम निर्माण करणे), `assistant` (मॉडेल) किंवा `user` (अंतिम वापरकर्ता) यापैकी कोणतेही असू शकते. फंक्शन कॉलसाठी, आपण हे `user` आणि उदाहरण प्रश्न म्हणून नियुक्त करू. 


In [ ]:
messages= [ {"role": "user", "content": "Find me a good course for a beginner student to learn Azure."} ]

### फंक्शन्स तयार करणे.

पुढे आपण एक फंक्शन आणि त्या फंक्शनचे पैरामीटर्स परिभाषित करू. येथे आपण `search_courses` नावाचे फक्त एक फंक्शन वापरणार आहोत पण तुम्ही अनेक फंक्शन्स तयार करू शकता.

**महत्त्वाचे** : फंक्शन्स LLM कडे पाठविल्या जाणाऱ्या सिस्टीम संदेशात समाविष्ट असतात आणि ते उपलब्ध टोकन्सच्या संख्येमध्ये गणले जातात.


In [ ]:
# The Responses API uses a flat tool format: name/description/parameters at the top level
functions = [
   {
      "type":"function",
      "name":"search_courses",
      "description":"Retrieves courses from the search index based on the parameters provided",
      "parameters":{
         "type":"object",
         "properties":{
            "role":{
               "type":"string",
               "description":"The role of the learner (i.e. developer, data scientist, student, etc.)"
            },
            "product":{
               "type":"string",
               "description":"The product that the lesson is covering (i.e. Azure, Power BI, etc.)"
            },
            "level":{
               "type":"string",
               "description":"The level of experience the learner has prior to taking the course (i.e. beginner, intermediate, advanced)"
            }
         },
         "required":[
            "role"
         ]
      }
   }
]


**परिभाषा** 

फंक्शन परिभाषा रचना अनेक स्तरांमध्ये असते, प्रत्येक स्तराची स्वतःची वैशिष्ट्ये असतात. येथे ह्या नेस्टेड रचनेचे विखंडन दिले आहे:

**वरच्या स्तरावरील फंक्शन वैशिष्ट्ये:**

`name` - त्या फंक्शनचे नाव जे आपल्याला कॉल करायचे आहे.

`description` - फंक्शन कसे कार्य करते याचे वर्णन. येथे स्पष्ट आणि ठराविक असणे महत्त्वाचे आहे.

`parameters` - त्या मूल्यांची आणि स्वरूपाची यादी जी तुम्हाला मॉडेलने त्याच्या प्रतिसादात तयार करायची आहे.

**पॅरामीटर्स ऑब्जेक्टची वैशिष्ट्ये:**

`type` - पॅरामीटर्स ऑब्जेक्टचा डेटा प्रकार (साधारणपणे "object")

`properties` - विशिष्ट मूल्यांची यादी जी मॉडेलने त्याच्या प्रतिसादासाठी वापरणार आहे.

**वैयक्तिक पॅरामीटर वैशिष्ट्ये:**

`name` - प्रॉपर्टी कीने अप्रत्यक्षरित्या निश्चित केलेले (उदा., "role", "product", "level")

`type` - या विशिष्ट पॅरामीटरचा डेटा प्रकार (उदा., "string", "number", "boolean")

`description` - त्या विशिष्ट पॅरामीटरचे वर्णन

**ऐच्छिक वैशिष्ट्ये:**

`required` - अशा पॅरामीटर्सची यादी जी फंक्शन कॉल पूर्ण होण्यासाठी आवश्यक आहेत.


### फंक्शन कॉल करणे 
फंक्शन परिभाषित केल्यानंतर, आपल्याला आता ते Chat Completion API कडे कॉलमध्ये समाविष्ट करावे लागेल. आपण हे विनंतीमध्ये `functions` जोडून करतो. या प्रकरणात `functions=functions` असे आहे. 

`function_call` ला `auto` सेट करण्याचा पर्याय देखील आहे. याचा अर्थ आपण थेट फंक्शन नियुक्त करण्याऐवजी वापरकर्त्याच्या संदेशावरून कोणते फंक्शन कॉल करायचे ते LLM ठरवू देतो.


In [ ]:
response = client.responses.create(model=deployment, 
                                        input=messages,
                                        tools=functions, 
                                        tool_choice="auto",
                                        store=False) 

print(response.output)


आता आपण प्रतिसाद पाहू आणि तो कसा स्वरूपित केला आहे ते पाहू: 

{
  "role": "assistant",
  "function_call": {
    "name": "search_courses",
    "arguments": "{\n  \"role\": \"student\",\n  \"product\": \"Azure\",\n  \"level\": \"beginner\"\n}"
  }
}

तुम्ही पाहू शकता की फंक्शनचे नाव कॉल केले आहे आणि वापरकर्त्याच्या संदेशातून, LLM ला फंक्शनच्या आर्ग्युमेंट्सशी जुळणारी डेटा सापडली आहे. 


## 3. ऍप्लिकेशनमध्ये फंक्शन कॉल्स समाकलित करणे.


LLM कडून फॉरमॅट केलेला प्रतिसाद आपण तपासल्यानंतर, आता आपण तो ऍप्लिकेशनमध्ये समाकलित करू शकतो.

### प्रवाह व्यवस्थापन

हे आपल्या ऍप्लिकेशनमध्ये समाकलित करण्यासाठी, खालील पावले घेऊया:

प्रथम, OpenAI सेवांना कॉल करूया आणि प्रतिसाद संदेश `response_message` नावाच्या व्हेरिएबलमध्ये साठवूया.


In [ ]:
# Extract the function call items from the response output
tool_calls = [item for item in response.output if item.type == "function_call"]


आता आपण तो फंक्शन परिभाषित करू ज्यामुळे Microsoft Learn API कॉल करून कोर्सची यादी मिळवू: 


In [ ]:
import requests

def search_courses(role, product, level):
    url = "https://learn.microsoft.com/api/catalog/"
    params = {
        "role": role,
        "product": product,
        "level": level
    }
    response = requests.get(url, params=params)
    modules = response.json()["modules"]
    results = []
    for module in modules[:5]:
        title = module["title"]
        url = module["url"]
        results.append({"title": title, "url": url})
    return str(results)



एक सर्वोत्तम पद्धती म्हणून, आपण नंतर पाहू की मॉडेलाला एखादी फंक्शन कॉल करायची आहे का. त्यानंतर, आपण उपलब्ध फंक्शन्सपैकी एक तयार करू आणि कॉल होणाऱ्या फंक्शनशी जुळवून घेऊ.
नंतर आपण फंक्शनचे आर्ग्युमेंट्स घेऊन त्यांना LLM कडून आलेल्या आर्ग्युमेंट्सशी जुळवू.

अखेरीस, आपण फंक्शन कॉल मेसेज आणि `search_courses` मेसेजने परत केलेल्या मूल्यांना जोडू. हे LLM ला सर्व माहिती देते ज्यामुळे ते
वापरकर्त्याला नैसर्गिक भाषेत प्रतिसाद देऊ शकते.


In [ ]:
# Check if the model wants to call a function
if tool_calls:
    tool_call = tool_calls[0]
    print("Recommended Function call:")
    print(tool_call.name)
    print()

    # Call the function. 
    function_name = tool_call.name

    available_functions = {
            "search_courses": search_courses,
    }
    function_to_call = available_functions[function_name] 

    function_args = json.loads(tool_call.arguments)
    function_response = function_to_call(**function_args)

    print("Output of function call:")
    print(function_response)
    print(type(function_response))


    # Add the model's function call item(s) and our function result to the conversation.
    # The Responses API represents tool results as `function_call_output` items.
    messages.extend(response.output)  # adding the model's function_call item(s)
    messages.append( # adding function response to messages
        {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": function_response,
        }
    )


आता आपण अद्यतनित संदेश LLM कडे पाठवणार आहोत जेणेकरून आपल्याला API JSON स्वरूपातील प्रतिसादाऐवजी नैसर्गिक भाषा प्रतिसाद प्राप्त होईल.


In [ ]:
print("Messages in next request:")
print(messages)
print()

second_response = client.responses.create(
    input=messages,
    model=deployment,
    tools=functions,
    tool_choice="auto",
    temperature=0,
    store=False,
        )  # get a new response from the model where it can see the function response


print(second_response.output_text)


## कोड आव्हान 

छान काम! OpenAI फंक्शन कॉलिंग चे तुमचे शिक्षण पुढे चालू ठेवण्यासाठी तुम्ही बनवू शकता: https://learn.microsoft.com/training/support/catalog-api-developer-reference?WT.mc_id=academic-105485-koreyst 
 - फंक्शनचे अधिक पॅरामीटर्स जे शिकणाऱ्यांना अधिक कोर्स शोधण्यासाठी मदत करू शकतात. उपलब्ध API पॅरामीटर्स येथे पाहता येतील: 
 - एक वेगळा फंक्शन कॉल तयार करा जो शिकणाऱ्यांकडून त्यांची मातृभाषा सारखी अधिक माहिती घेईल 
 - जेव्हा फंक्शन कॉल आणि/किंवा API कॉल योग्य कोर्स परत करत नाही, तेव्हा त्रुटी हाताळणी तयार करा 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
हा दस्तऐवज AI भाषांतर सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) चा वापर करून अनुवादित केला आहे. जरी आम्ही अचूकतेसाठी प्रयत्न करतो, तरी कृपया लक्षात घ्या की स्वयंचलित भाषांतरांमध्ये त्रुटी किंवा अचूकतेची कमतरता असू शकते. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत मानला पाहिजे. महत्त्वाची माहिती असल्यास, व्यावसायिक मानवी भाषांतराची शिफारस केली जाते. या भाषांतराच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमज किंवा चुकीच्या अर्थलावणीसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
